# 70 Job · Tarea 01 · Ingesta Bronze

Esta es la primera tarea del Lakeflow Job: parametrizar la ingesta con widgets, construir una tabla Bronze y publicar el resultado como un *task value*. 

In [0]:
CATALOG = "big_data_ii_2025"
SCHEMA  = "spark_examples"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Trabajando en {CATALOG}.{SCHEMA}")

In [0]:
# Declara parámetros con valores por defecto para ejecutar el notebook dentro o fuera del job.
dbutils.widgets.text("entorno", "dev", "Entorno")
dbutils.widgets.text("limite_filas", "50000", "Límite de filas")
# Recupera y valida los valores recibidos antes de iniciar cualquier lectura.
entorno = dbutils.widgets.get("entorno")
limite  = int(dbutils.widgets.get("limite_filas"))
assert limite > 0, f"limite_filas debe ser positivo; se recibió {limite}"
print(f"Parámetros: entorno={entorno}, limite_filas={limite}")

In [0]:
from pyspark.sql import functions as F

# Define el origen principal en el Volume y una tabla de respaldo en Unity Catalog.
RUTA_VOLUMEN = f"/Volumes/{CATALOG}/{SCHEMA}/spark_data"
RUTA_RATINGS = f"{RUTA_VOLUMEN}/ratings.csv"
TABLA_FALLBACK = f"{CATALOG}.{SCHEMA}.movielens_ratings_raw"

# Encapsula la comprobación de archivos para poder aplicar la cascada de resolución.
def existe_archivo(ruta):
    try:
        dbutils.fs.ls(ruta)
        return True
    except Exception:
        return False

# Prioriza el CSV; si no existe, intenta la tabla raw y falla de forma explícita como último recurso.
if existe_archivo(RUTA_RATINGS):
    ratings = (spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(RUTA_RATINGS)
        .select("*", F.col("_metadata.file_path").alias("_source_file")))
    origen = RUTA_RATINGS
elif spark.catalog.tableExists(TABLA_FALLBACK):
    ratings = spark.table(TABLA_FALLBACK).withColumn("_source_file", F.lit(TABLA_FALLBACK))
    origen = TABLA_FALLBACK
else:
    raise FileNotFoundError(
        f"No se encontró el dataset MovieLens.\n"
        f"Se buscó en:\n"
        f"  1. Volumen : {RUTA_VOLUMEN}\n"
        f"  2. Tabla   : {TABLA_FALLBACK}\n"
        f"Cargá el dataset antes de continuar (notebook de ingesta de MovieLens) "
        f"o corregí la ruta del volumen en la celda de configuración."
    )

# Limita la carga y agrega metadatos técnicos para trazabilidad de la ingesta.
bronze = (ratings.limit(limite)
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_entorno", F.lit(entorno)))
print(f"Origen resuelto: {origen}")

In [0]:
TABLA_BRONZE = f"{CATALOG}.{SCHEMA}.movielens_bronze"
# Reemplaza la tabla Bronze completa para que una reejecución produzca un estado consistente.
(bronze.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_BRONZE))

# Cuenta la tabla ya persistida y publica el total para las tareas dependientes.
filas = spark.table(TABLA_BRONZE).count()
try:
    dbutils.jobs.taskValues.set(key="filas_ingeridas", value=filas)
except Exception as e:
    print(f"Ejecución fuera de un job: no se publicó el task value ({e})")
print(f"filas_ingeridas = {filas}")
# Muestra una muestra pequeña para verificar esquema y metadatos sin recolectar toda la tabla.
display(spark.table(TABLA_BRONZE).limit(10))

## Cierre

- Verificaste la resolución explícita del origen MovieLens, sin datos sintéticos.
- Construiste `movielens_bronze` con metadatos de ingesta y del entorno.
- Limitaste el volumen mediante un parámetro reutilizable del job.
- Publicaste `filas_ingeridas` para las tareas dependientes.